# AF × LVH joint-head analysis — "detects AF, misses LVH"?

**The question:** the previous round (af_lvh_subgroup_v1) showed the LVH head
deteriorates inside the AF stratum of the frozen test split. This round zooms
into the **28 ECGs where AF and LVH co-occur** and asks head by head:

> **"When AF and LVH coexist, does the model correctly detect AF but
> frequently miss LVH?"**

**Same frozen everything** — the SAME `test_predictions.csv` of
`resnet50_hires`, the SAME frozen val-tuned thresholds (**AF 0.81, LVH
0.47**), the SAME ground-truth subgroup definition. Nothing is retrained,
re-thresholded, re-predicted or re-split; the frozen results folders are
read-only for this round.

**What the notebook produces** (all written to
`cardiosentry_runs/af_lvh_joint_v1/` on Drive):

1. cross-checks: the frozen decisions reproduce the saved predictions
   exactly, the pooled AF/LVH rows equal the frozen reference numbers, and
   the subgroup sizes match (refuses on any disagreement);
2. per-head evaluation of **AF+LVH**, **AF-only** and **LVH-only** —
   sensitivity, specificity, precision, F1, AUPRC, AUROC and TP/FP/FN/TN —
   where the subgroup is the positive class and the negatives are the frozen
   pool of ECGs that are true-negative for that head;
3. the **joint breakdown** of the 28 cases with bootstrap CIs;
4. the comparison with exact tests — **McNemar** (AF vs LVH on the same 28
   ECGs), **Fisher** (AF recall AF+LVH vs AF-only; LVH recall AF+LVH vs
   LVH-only) — plus recall CIs;
5. the **verdict**, applying the rule stated in the header of this notebook
   before any number, and the sample-size limitation with it.

**The verdict rule (frozen):** R1 McNemar on the joint breakdown (p < 0.05
in the claimed direction), R2 AF recall on AF+LVH not significantly below
AF-only (Fisher), R3 LVH recall on AF+LVH significantly below LVH-only
(Fisher). SUPPORTED needs R1 ✓ + R2 not significant + R3 significant.

**Limitation, stated regardless of verdict:** the subgroup has **28 ECGs**
(25 distinct patients per the previous round) — recall CIs are wide, the
cross-subgroup tests are low-power, and the verdict grades the evidence for
the **association** on this test split. No causal claim is made.

**Runtime:** pure pandas/numpy + exact combinatorial tests over 938 rows —
any Colab runtime, ~10 s, no GPU, no pip installs.

The whole harness (config.py, baseline.py, metrics.py, stats.py, analyze.py) is embedded in cell 1 — no folder upload needed.

## The reference-pool design — why "AF specificity" exists for a subgroup

All 28 AF+LVH ECGs are positive for both heads, so sensitivity is computable
inside the subgroup but specificity, precision, AUPRC and AUROC are not (no
negatives inside it). Each head is therefore evaluated with the subgroup as
the positive class and the frozen pool of test ECGs that are true-negative
for that head as the negatives (**653** non-AF for the AF head, **683**
non-LVH for the LVH head). Consequences, declared in the tables:

- specificity and FP/TN are **identical across the rows of one head** (same
  negatives, same frozen decisions);
- precision/AUPRC/AUROC with the 28-case positive class are **small-sample
  reads** — reported as asked, but the verdict reads are recall, the joint
  breakdown and the exact tests.

## Cell 1 — unpack the harness (no installs needed)

In [ ]:
import base64, io, tarfile, os
os.makedirs("/content/code", exist_ok=True)
_PKG = "H4sIANpeiGoC/+19a1fjSJJof9avyNWcvdhdsrCNzcPd1C5NmS7uUMABquf2Mj5e2ZJBgy15JRuKZtjffuORmUo9zKP6sWd28DndhaTMyMjIyMjIiMhId91d//dT78vHwPOD5Jvf5dfk36p/m82NTvY3vm812632N+LLN3/Ab5kuvASa/+af89feFrNFOAt2W1vbWxvtdrPZcrfa21ub29vWN2+///W/cRxNwit3fv87toGTepPne2urm/sX/2y3251vWt12p93sbsEDzP/O1tbWN6L5R87/aRj5izS4/3IbhlXlnvv+D/qzbdvaOxBfxNFPH8Xf4jBaNK5hKRBe5E3v0zAVDZFee0ngC+AUIFW0SF3L+svHvQtx8fHwXJydfD7+IA7PLessXkZQbzKc3l4P0+XoKomX8+FtS6SLxFuEkxBgLK4DMUniX4JILIJ0IdL5NFyI0b3AwpHfWCTLxbU1jmdxMgr9cHEPePiAQHwnKyOWhJ8fLIIkjAFykIowSkM/oALQF2pvObNqe5/PTvZF093Z3NwQjffw1/bOVscRt0Hih+OFOP98enpydtH/UHfFxTX0lZAQv8TxDGEuYoLY3rb6+z+m4u46SAg+ooR4jONGPB4vE3rhpTcpFUfsGqN7puJ/LaGXYRz1LEvAD0BEeQjBlzBdOMKPA649i/1gCu+TJBgvpvfUTcB074Dqj5YLMQvTFCv/m2WdezNNzgA6db+4DqOrHgE63/vUJxoP5zB40FvAInXH6a2IJ+I/kyCNgkW3ObwO4c//dCxdRYK79aaNxTIiqkOJ63jqp6IGqAMNWw4h33Q7W3Una0wOoUVjKBQDQA8mYRRi8644jglDgaQOYJRCaMCBPxu6EXi24FniDM3HCX4nPsFBwgF4h61r+NCdaq669lILVjcau4bwkC2W48Uy8aZi4gFNZUWg9XyZOvDZw/ZCSUqx8EbTAFj9Q//88MfjngAeFDZy1zwYAzOPgTltGkh8uff59GzfFjSaYgI4expBGKYA2HMR3kKNdRhWbzpFAgCTz5fUiDg8Pj/80CdsdLdqWAzQ94jnGtiSBf1uzGOCFQDLnmeYOAIoNg5ToLLDyOA/yP1RAJ2KgisPK0E3EWNoyNINwbRewECkIoqjgEmcBBNg9WgM4xDHUxjBNLyKetSvwBtfE4s7OShrKdMZscVuKCzFeOoBu8pGMzx0QTls2I6FA4LDR5AW196CSyXLoKEqioOTM5A7IHw+9vc+iNpmdwPxbsAQIHYsACzGb3ObvyG3qI9KfNCEhwZm3g0gwwOeBDRbgVwz4M1wbBkjNA8SPTRAw0WO5fQEccRdCKxP44hTU2FtERmJfNSSpn0CPAXyleUMPKTCDydAe3FyfPQzsuLddQj0VtRMcZrew4gtI5gM+zChEWUYJ2jZD4DUMGtI0nxrcilR/+B0/eKY6AmcdnxxuL93JLxxEqdp1jbQH1iAxWsNO0Ayx+Ad6pQv+SytA+qED0JNgnmc0HwFdIhc2DfoMsFASk5BkkVBmrEgzzAAep8CCb4jtDUXUyFiZKapB3OhMfbSMmtB47NgfO1FIc6se2ovneHsqU2CO0Dk4lR4V8jhi2xkDk6J5aq74KUozVESKZmL9dSikQB5UlkBW2QJSCunGMHHGz++i3Qfgy8gaggOsjZ0PooJHHYkCUDK0jSj4tFyFsDCFSeNVAoMFE7T5SzCFffiY1/81D/7cLh/Ic4+H/Whd8x+DSW5cHkC4s5B+LEcoyX8lwB0O2gWOc/y/L8ticNhaZuE3C9odhQkad2yPv58egLNnB+e94T9l1VL1epVKsXyQC5rQlMpwi+4WgW0Xrk2qAgtIf5yCHrDcWN/77wv9s5//vSpf3H2M/Ti0/g4mHkJSBKiGYuCOKqkbjxhktLqLJAtYHHywxQw8kFDEXMvTNIeSWaJpHhHvSB0fFvcpgQAC/Ar+V2WtlG6gszDGYTwYMLPxfew4jW7SFhaNqYe7Jp8AuOHWAm5dpcZ+y6mWZSKUXDtwTDyrGaKnBwLHEtaMlHSudZZWyDljk8uxIf+j2d7H/ofgB7wRi4WcaQWvVskcSOOpjCiB2EKIy6JBfO2gSqQ73K/9iuGxqZVBxdWnHbhpLKFENcBZtg0IwDAGQVTmNCydVkPUN8QRDcDb3ysQhwXrxdg/kkzDOHr5RUJkGeCEAD0KxsK05W4KwQ08lBSa4CCf8CgRtXCWGfj7AgYM0Wn/M+oDqU2zGfs4ene2cXh3hFI98oGUdgAaPMVrBpnGyQ2CqCQX877AHCP8QdQhVI4JyTGoPQEt0BmlM5IhDpC2D85voBhA5HCBKjufTwnkRtk3QfSAfueHvUb54f/0RdHh58OL/YuDoGza0UtioUq6nhXXuJPQf7jMiNlab2XV0VAYxNKY6uRGgEiMoyXSjGnhQ+gt7uAC+jV0Rin+iKEiZWCJprGluSH/UOW0He4K0AJdtZeByLiK+AE0GpwXcBFyUvCFPViUnuUhL9KPJ/X2oxmWrs4Pz/ZP+TOknACHs10TtRwQRwtUyA3MQ2y8AygWQ0WRvBJKfrA5za8iHDMRijpo/F0CfMA1YqZXCKkbHaY50klQoKpjUeCujRLPyloCJ87716Nnd4ceNOlx0OHGz5qYjicLEEdDoZDEc5wlAQhQ8VSyzra+6F/dA4S7dI+v+h/OrQd7AL+H+cm/HN8cvZp78geWNafiHyZ1FtkeylaLfWSHyZP7i8aAAl2BVE6TsKRogR8XU4X7t9gnNQGpLmT++ntyGb+fR3AYcOk5jTG18H4Bld3QxeY4ho7CUEcju6N5dK1ULvEvteMLtdhDT7rn388OaJPD2oFsXuVONHvT8DCYSSVCtw6qYHRKracIQwMm+lV9MSpAtbZsh4V7dUUIkVwgtsq7NGPtDlvXJx9vvgoph6IwXTldslFYtHeHXmWdUIYQFiOAfbfYN5/h5+JD5GmuMfnB5hT4SKVswk+pd8hpOl06E3EOv2BMKGXSewvxyWtP/BNWqAaWjPGC8TU+ecfsB+nBs0ZT6CUEA82SIL51Lu3ebnnZbwW7ba36zaTjYeo5Sjyth4dDQa7gHAKYHidAG5DfQ0h1iX3m3CaCo6iBbzLwdELTg2ASFg0lQBaeeWQnJShCX/xxDOaYrqWe45CD3pfK1LUwNsEIalXAkG0q4SRkY54Dr0FaFHJbVlp9NBWE5G4DEHsA7fLJmAd84OEZ9bw7OQvxSkEU00Oq5ONjKN7XHfMGWKW1dR3sr7VJZoHzGcawzT8JdBTwDAVwGYbNmFJ0AAUQST4LBhIdrDUMPhweIyYZzzY3nZMVmp3t5wcS7TbW8XRzoaxvd11zCFpd7uP1kX//IJa2dnYxl78EMcLtGnRXv0WFjsU0LxzAYkQ4+Rcg4m/jHAXdIWa1PEQBBWt0EjmVrvZtM77sMbvik7b2js6/bgHf6I2i+DPsum3jASudwuyKDwjNg25DHpg/6B/1j/e7w8/nXzoHwF0O29iso0ip2cnFyf7J0dD2IVBwY3Nne3tcTdojryuv9X27Te/wx/4c9/8f2/+v8z/19rpbrrbGzsbWxudt3n4T/AbwfZhCgrj7+kBfNr/19zCv5X/r9Xpwvzvbmw03/x/f5T/D3cQk3iZsM9KK36O1NMdw7a+2lFIXsGfxce9sw+NfVACPljW4aSw14EN/NTH3S7s08m4rh08hgOI92L30lPEe2zr2kvYeFxysWXb+Aw8brnRxplehxNyL5JmhPBAO7XyON3BZkqqfWQGyLbd0v2Q7UdPQHtxctZmh7bS7NwDVWwRL+7nQYNVo3T9+n6UhP7wtrWeV4bWDd3JsuTGyAcFDvBgMGP0FwC+QUJbPlHUkRxlGTV8dhaoi41w5l3lVNsKM0qt7KOti2WKrji0tVvGjpy36LItspzDxj30yUag9o1WtnGWxnHytBz+1DeoXDO38e+qHZNEShpfRTfSSCfIFQki8QFHqV7c0yv/DY8aG/sspa6idzkA9g1474wsoHirJz1VtNsxLN24jXHEKM5RIvNa5DcTrtiL7nGP410lQTAD3Vmki3gud9BLNM971iy+xYaNecBmHw80dvIVSsvTy40zfxKNX/nLz8zXKfAXe+d/xhIzGNGwQYYFUaO9qkP7XKCiI9g8VH+V4p8VPTw+/Xwhy22OdkYt39+c7HRbI89vmuV+OuzTVvJya3N7YGz5quxLtB2VZi3aja5E2tzQ5Iw+l023s5G3zcAGrsL8U2EZ5l+Vcafpbmzn321qq1oml8npskZOFnYLqXlZsuU4JSsaADP3awRZ7fPjO5GbwXITnGpDpDAmKlnLANoMlk0R/NfSm7K1j4z503uTcIBnZrFZJvGYTFsw5KBabnc3drabvHVezhP1qd1tdTa3kC12Wl25bbbTzH3OxTrNjW5ze2urtdPGKgDD9MlTka2tZnunubOJcUQ7nW0JSXsXJRwYiM3O5mZno9Xc6XYAzqQlv2xsbrZ2tlrbG82d7U1ZezJP6Guz3d7Zau4AEu02aCztJm74NZdVWwNlT5ZznMp63494gNDG6nN8uYl/TvDPFn6e4PcWQce/NqFkZvEr8wX7r5AxpKc2hSLjwBV9Hib2MivOmONwAixpj8wvEQDN0AtpAcXgH4yfCPPbfWy0apQ325ut7g7s40C/29nq5se53YFha3bg383WZqezYpy3O1vN7vZ2G+i802m1tqvGuQsbxR0Yqo32ZrfZrB7n7U1jaLe73U7H+K84tp2dre3mxnZzE5h0Z6ddHtrS7C0NbbdqaFubamg3OmpoN3b00HZ28hYrnMsc9CKughjjEe7R0ZtbepxsUSbHSJobSJzxcijpKy5A5HBPtaPbzVua9g5wNOkRNoX696f8qmfY9skhnJKTH+piyAJX7m5ldfcOdjlcaLeJ4StGqfaWLoSfWyiFd5sOW0ih9N7REYgQiU6XHjV63a6aBjowBKWZpECgFFm0R0rXR6AMWD0V3iMjTcirgvMgLUSasM7rLUjbdLGHx/0fh6cnsHwhDpJ0DaHxNCi22d2g3q6uwV3JKmxvWJblBxM2OA5ZpA95XGt1NLUex1HQY2az7f1M/cnNapT3SntlRz2pJDrGYRp76GpFNQMhYR9h+Ub33MTByAoUBzXp9DAluaO7WV+9tKENl31DOfngZB2u13u6tpemQYLhFJNLnCMD0Arpb5gaA7G7i9jAxsN+APweQVWbv5tE4l92s1gYuxKSmocVIOSnFwCRGPCAGRAIgQd+/Wg/35Nir+TfiwL8v+YoqluTk9qP2a27nKEIz7cuW9Yz950wJuKuHjPyAZB7/J324ENP2GnAkGgSZnQR8XJREVJJPCSXehVPxeBUzCXDIuctTWQBewUP1xLQeZMkBMi/BElcaAyj8zgMjyLz2JWbyR50LBJ8T8BOb0GxhIQ8yIE8JHcFVUzJs5uxI/s1FGxJ6HwTyM8m0Cq98LJFw2nOl0tj1RgQ+TM9VKTTePE80HYBKGBShIrIF8FaVqUEsd7sv2/23xef/+i2QLveerP//hP8ONY1/V0PgDxz/qPT3dgqnP/otlutN/vvH2X/PVUacxYnxMqzsnkV4i+dzDSgIjkMUx7HquqFq4EKsOAoelFLAwp14iNH2q48WcJSm3gcqBbULesAPhBGH/kAiNp9nPOmJ48n6O4U9ZwpFLu5PYsKTldKKq7rOrJchxMjLh/rlhHwDGAKewUO74ZqQ9z1NhFYIZzFkQ70sTct6Mkrwr8pGEHuMrB1HV4toIV5Eo+gqfe7pVhzy+pXBq2jtXSKQ6dHiO0+FCkGdF9ScDVrBa64uIst1LjY1G0ckECiq4hyZSLg3aOKL68800DhgqgOSmSoZxyVkBrxzXdhSts0DtWU1PhOxqjXrslEamz16xxkpOi6Im6dgBHPjAKKs8udOKDBoW5hsFJGZKn0MY6FGHR5kkKGouNhiiy8Xw2hl1YEpGcx6A3Yns+nAdvFezJguTKOfRYEOAy4dVyH3SCVycXfUw/IcrciXP27p8PUdcT0k6HqHKYu5/D/PTk8vhA/nPX3/vzh5C/HPC1l0LXcUHD0Icf10tkQ6hsGGmOv+NDX6J6s6TI4kGMYw8QqHZ8g1moKUYralk/wtVXxlQO44SOHUct47lLVjYqvsupLre7yHTAyyC4gezRXr+ZARBoHMfd1OZZz+HJfbu6Hfnhb8xwxkvvgJIDWYATEuhhhSPNIBFMg3wS26YuaHXmRXVc1yXxWw4M+si4hnN5MAy+JXLmKa6yBC7yrYKhZeYih8YHZJrexoiC3U0eUpkFUi+buMgpB0NTu4eV7GIVsy/oEwkn8YoShKFQYr0Iz9/lZ5FahpP1dM5gvNX/Sg8FyP8DW7yDxZoEjboL7HgpBMrfAp/MAN6za5vKjcURRIIjCQS+UQ3Jp0zGL2tKC4gwE+n726RKaG9C3GUp61VztApYX9FT4wZddf+LSH/W8tQZWPxTKCNMNF8EM9neZXQXJ4o1w75hF9+WWIjzwFUbLQL+cif+zK4AelxOb1jayP9gD10vRm1kDaVFHcNBq3RyamSQrTushCOIKiuInIqmTET9HZRRVmsAnUXFl72XEzRb32+yc5VqqFkZ91Io8sbQQy1Na+XJa2uiRgTc0MMbWG6vIocFh3i1xTr47ddkoKgxARSwzUKKbXxm0RdgVxG0O2Im8mEMNeFmrATyoRBqAqoSKAKBbd9PlrFaXTBE9VeH7cnndAGD3kgYW0VMV8g0wq4NqAFVI1i3mDnbpHaBZN+cBf4QVdoHe4MlckjAxPuqa8uOkpT615SIt1RA++wgl8YHEQi334j2sKSQSmm7TkmalxItuUCWSYqiXW9ArDjBKnsutxQTqHpACCYRGrbSG4sifSLrdEyO4yM9BOsAwTHp9EwQ4ArUXsERdVhZ/FwYocwo+6DlsIxC7x0czs7eqW3Yvx7FGiWgIXYXPOMKIPzzVDRsvuy9wMNh7MZkr78UkUt4LGEejxZwXh8bH+Jjz3+CT8dH02uDfxidy3kxa5hty2BA7IEoT5JRFZCKu/E1q5bxEyg8cmI7uNB7TE5p2c+wsSZwHwz4ttZ59JRjTiQR/5wYAuGqIDGWMArxSo/AoxSypbUOttpWkLcnTxRK0zUtgSFBHksSDlRIl7CDzG3iL4CpO0J/n8zFNQ49DEqJy96WtDz/TqSDiNYzLhzrodcJvdXkcnt6Rmtl03Q19ZAi961T5EiOjbyjcJL5xRPaEqhc94x/md/2Mfwzo3DOJITosSifCSL9V5xjiZaK2M6qTalvBsrcsu1Xsdp2lNOGEUxJKXvJA7h2wLMytCugVGag5aU5WgsI9yINB43EFHHq9EhBTlKQKkbgm8QMBssL1oksqHLAoniRo1V9Vo+2IDSX0idq7poAxB1Jyak3iSrKKlwCnqgKddSpWaVVXMdihVKX9TJXKdjbylR5N+WlytPWbRNWAjjbzFrjdtvCUw3D/hM9QgSb6Dk8osLitO6JmX5za7B6mpwN6mqinY5vlbHEEoRp9WkRcEHML4LMpdvkDCFe74DIvAztNuFQme7n5ls0uc3ri5AbahV+GQptjLoDCEuYViywmxpAURKJxDffrPTEN08UlCSZDRST5RV/gQUosjEOCZVIIlqy8vW+sNPQ4OduNYZnJfHXoXkMkLpuDy7VM9K4NHhto4GHNMfOpPJT1wp7bnjyaB3hs/t87ED0uCukaNHHde7/zaJPGfu2IIcpFzQ5S7By53nweRH4tq9+wQaWp7cD/cA3IKtQz/T9BUERFjYAGNCFID+YeI7lc01awAfRY7gjWBr3v29uA4LuKc0hZL5LLmwF0xO1Ah1GvAg6hFDBjGEj45PAmq07a1WqvcAbIlyQZwl4rT5J6BUlqyMLrWeKCkgGIjD+je52fAkMh7CpIeb1NRsdI1tCmIFfgZFgnZl9ncw+KyGqA8rBlZs0p6I9hUtQgOdFANbRKcw6dBcYveM43Z9Vx63ZuI3YE041MNcPjvU/9/Dk5U2TbqwwrtmNVS+yVB+izCnl5ba8ywFTVMBqpMsvYFBJjChJSgWojkBd0erhCYES0WZsBe9qkFMDyq2fNyJdLrtrZaOlSTi9AY/sQPebNXLWKjB850QI9yUuMtb2DNZYYYp26VvgMr+T33JlAOZPXxlJdW6PpCs/Yp7Xe+y34m7IyraGceVjb6f6r2D+EhxZInafFy2YnEya4ExARTgYgj8FBZaMCbpFHfmayIEUlhJcT+xJm99o4HE7ju7XBt61ms9dxW5PHf3Xk++vw6lp+6NKHgV2IeADJguNFIDgywnykrZvdaNgrBR72gAmELRKJBkQjeJyPF7Lx99vU+sM4ZDLVX0SmYltri3jhTeV4RNQKAHc1+OLMfDv/9eb//4c6/7Xd3XS7G1tb25vtN///P8EPE3b8vt7/Z/3/zY7h/4dnzP+4ufmW//EP8//3M88feQNH+sA5JlJRTnrjzFd25sq1rOMTkY7D+X1PpXDihGOc5IcUd3SBYhxq4OvgwZoOSSU/KeZaG4UReodDUPBBhUtJ+6ZQQDa9xosA8LqhJDEUQ0h7kekUFPcf0LuIflfcnPhe4htJB5XDWuFWIwww8Re0F8+wNTPTVDGJlNIFqfOcD0ypiT1xBeo97CBQNwLFYWxWJoXRkUe67HxSKFRvCRSdnwBixJNFENnsPy3AQIvX3kFDatHrGG8pdWU6qiXPnY28UTjFVG+t9TZ59TmlkkBLM1rLT2v/T3y/Cw1HtZEjxnXxd/ED/O07WKFed5lEubxMNZ2YyRFrmLdnDWl2fT8PEhmRjv59g3LabMh2P4mxI5X5gfgiLrN9us4WMugRkhJBr84u9kn4RabWgiG7CtHGz+ZGT0UZGLEdaoPCafEsS6dLwBwa7WYTIyDIC48Z7DAbYqftoOlzjNvJaSBYeS4kVLCyVAuuOMuyCilYPFgKhQ/9/cPzw5Pjc/Lbob7Iub6Qyy1OzEK8DoPLew2MR1GsnQScdYsYLoyu6t8Z003tALBxSzduJj5zX+u+hp3UdaUvu9pv/RsYxYzIAtrOzcYRzsXhvDYiax3wA/1LOzqyKWiDdcbLDERN4zmJJXOycLQ087d27kVqblpyoxGRtdLY2LCi3nKb7J0RxiyRltBwKjeVSDgXBVUtckRY5yQnZI/B4JHaDbTTquddpACq7TZhbhGYdYG+q29FBGINWlR+6QnNuyFOMiCJJ0mSJ40j/FUkOlETNT+BMU/ldBEkEVk+dnkOi9o4wOgnlroBnXCIWBZ9mQeYcLVOR0svLzFKYeDAPIaWBwOelkl8t44nU0k817x3SKV3Ph/IhEcoOoLHjPgt2F22gXgeUAbLwj++tHFPW/L9WI9T1ILHqM3PS3SKN/VO9UtG5qZD9Vp5lzdBfL8LBSmVIkxm9QbANsSXvAscwb/bFdmAYmFHfKnDSBmjLKE4EkZuaBHEusixRNSq/zbzRa+/HEkxDmu33jTticyvYzh7iBmkKU5aQbC0IId9evnfUClMIy8iGNL2KEMn6E31hDDDJ5x8MIVRqsYfoIlMnhJU4O8msv2+y9lm1kW7XrAbP1e1hjTP1a+rCcMRTUgXbb0ziQMjMdQyn2YNOfiNpDhFtwguC1lBTJVTjk44JcGN6ZxneJzoHUhkVA6yDISYO1UtCKjLJPeGcVHNCZ883ICrlxKuWQ/qJU9QEl1xWVwo45kLXfeW08UQ3mMcpc+lQv8LlIJ3LuYGuwqSFKcIuaKDMepP4S/Bbs2gSPZRCis/8e4odiEYXwK0gTsLgF28L2G625LOrdgR12juIZpj8RwbPNhMBXTS0qgCJAYCgy6UKadHYGxpEkJfdZgfBzo0pzyghN9jfsD9YLrw5LAPvfyQ46tR7lWlRfq1nPFy/viAyIEYRM2yTmojKHy89q/JDqSOqR8oXQLjENATy6oRgSNWCiIOl0TnSODNBGeoSpZRWsimxvIeRz/BCaR5zStx2tArc9moXGr01bzorWBFbzUjeooNw9GKyqPVlUeah3E5uQy9PPPiWFyGo2c5usDNxGeamT3JygjsK7gafX6ewdge1Y6GI+PdqCgchZxRwzIaWFt+HOmPGrEV0QLYS04A+OsE5VdITBKUOnW43E51pM1dqrchjqeOWJaarcxYKKMQFPaEOIXamaEMKYcfqK1qltycNGTxiSKgabfHPK83uBIfmE6o+nL8uhlGQKF05vxgPH57YQ1wV7M5fpSMHi+R5A+PmRE/M+EHMsQ6qNXyvh+n5Nt56hRnlWvHqXDdmKc51RqCqNIigmrFTb088Z5aTmT/LrE/A8qyx04c6dKXYY035SiAPOrz8UJPDKPSS6ZuLjoAUPnfbwJ9s/+/2f9z+d+2ttztZnOn0+q+2f//CX5Z6pf/Kft/Z6O52c3yv23J839v+d/+MPv/xTOp3QpJoytSxYGSdvb5WJycfeif8QE7mXXLi3RiLoAj81dToHjLpWwUxSxmbFM2Mw5VZDBbptI8LGorc6wZINar0o3V8bRX20XVT8byZKfCCrfTwHaMDm9lKeoymzumNMBkSJa0npgbs9S7pbKBP1Q3eWQOkCZaxmfeYnyNkcL5XGQMzExIlk+4lOV8MrP5UJBLlraHb+hhhauQsJehUcqyVYnNuAjhh4TacLNMKvoOoNy1OJg4OgvLWxmSxwhVH96Qd7Ksul4H8ei4YrQMp/6Ka0EqT6epU385txYC67qUoo28M9CKPKanwOiYq+U0yO4nyV1Jwp0xzsVKCyzdR6FMQ7XS5R11syB8UrxTLK6ya+DWkwywecfcKFApOmbYm02XLn+5LyOPBOA8gjJjEOwsGjjKwBizkP0D3ADl83vRLV3i6Vu6ZEpCmQlR5vBT5xE9n5OG8HZN5ROClj+f7/3Yl9kU7xfXsXmHjWg0TKnwfdWcfl+ILMp+lw2ZlbFB/P+9MRneD1bWajRwu/U9MMnQD5P3r/W0eMnV3EvSQD1Tvkeq78M8Qq1L1VbPMGfh/7/gcVYqN/cW19NwpIqdrnLdPHUMUYmEKveOcgnJ03jw7pN6R4EBliXvHBrSnUO7tC9+7VVElJFRXkEkClcQ0VLwm108hIcMV1w9RLebPHf5kL6T5mnxIrllxb1DT147VLpySMF67cVDVZcOSVivuHpIiK+5fEjUTE+T2oZrh3GdRYA9rryE6NqbTlbeRKRB5e8jetFdREJ8xW1EL+zKzLiVSHXg6auJLGEK9VyHXno5Ef5++wuKXnBJEf5+w4uK8PfrLyvC36+4sEiIZ64seuKGIlG4oQiAmQ588Ypbh3I3DQEcvmtIfMVdQ99hyADfNQRwcrcNiRW3DcnJhHquTwd8ylcNcTQB25BRPTdX2BquRj1ahNQRaX36rKcd7JRCa3HtcnbAXBCxh3kfzu9B/M/6X8JFbWJ/Ojw/Pzz+sZRhtCceEMjjXyP7qYMFmdsaJeWLtgJPwqu+L1UmHnal79Of8Hlt1GOGUIDIUpcHfgOf7JaY3zrxbTLVT4dYgM7iUEJOPpUDI2g/mse6kXv3XU5Pm9GMIP4dQObOZmcnDo1H2GXwI4PFIQ/ZKI0wYG0OFjV/4sqNSF0NmCz35DAVh4cYTcKX8GDEUjrAUJMf6irEGg/k+piaj/o8wMxzRGhY62B3WKuj4JAnZ/Hbvssp7l6HEE7MBwnlkXdM63T7J00Yem6IJ8dec5LaoT0oTB6FBpJ3f/sTlbQRz6MMKfFaxSl41jWHqP3xyQXxd8rmWHWAAd0mekNaTq5TSsCX5ZQ27vhSzgtkkWkww53X5UAdfYZ9d7Y3XbnlNXKAKzVOJpClihJWLvospEx+oRu4LKpwXNbSiiaMu0LlcWZjCuRu3Mp4AOQ7aTp8btrg/fL5Sng9KPtnaErM1FF2BvcvdEA0N3cGubPvaorM8uEjiq7m2QCZtPFhNnuUO37QuSjhoUoOzjss+0nHyyQ7KExuYewniraHUg9xdslRaLvFvbtxk5H2qvH1zIJv2GYl4BfalD/csH/lk1s+Kqt9LexqYhmlj5flnFB3cgXOvg+PK7NTULuXNyQIsNLztNW9e7hBEisAjwhB8usDQjJosuEa+a3TZwwnpVyoZhpUbVEx8zuu8j2pBKgVlfAWMXMxBKVvV3xyjewZDrevbocyM3mg4swHYfQVV5aZWOpmGDL1a3iglA+S8gFSeVa0VzpxE99dUi0aB8xLyk+9UsdWcrui8APVfNx9yGDS0NjP+BiR3dXwZQg8FrvG3TIPuBaOtebOr/K5VYczBFSNkjzFmp1WLZHGG6U16soA5BkhNqDEMkFj86uooygz6LnbE5NpX0IgpgxXtXNSSaFpJgWVCJuvJOo7LxRh5uFXhL6mn9cGL8dd4p2rnE3Ojlu2SJKITLMMW+UJikybraRKmc3SIpfLuFeg8tjqQo2hcaGGLVWNyrsJnqeUiXh2X8f8JUTJI7dWhdxa/QXTJ5s6K3pRnEe/Wra9Tr5R7GxyX54weJTQIMMl64KDS3uOMbAYXQEPuMKVan6FoDP4Ape4hIhOMBzRaNXrz8m+lzIBTx0lCJ+f1yZLZDjVcytatUjkWN1xMIcN85+D+36SxIkjLkDXoT/raMOD7y8TVGYXlhFuaGhbiAqHVGcAlJ64QEQF5AnlPB/Sd3B28h/944ZmE7F/dnJ+3tj/2N//szjYOzwiK03VJRPCj0Ee2IUIwVi6LTJBIcmVuaJoh61dV3d0J5C8ZLcAzRvF8nJ3rZKSgdEV5xcnpy5sPvE8KPzDJ+JV51WiI7kLuLTMDWnxWDnu9SkNnuFAyql2lWkKDD3QiPZTN92YfiXUMoveJGPdy6lP1cqhqRiyw2cVThxj31OmLKmIrXE6FZDwjrbJGZ/wmb6ZkLS5SxVUF2biOpHHPrugB1UmgxYr3WBqf2Q2WLs4xfs1xMGpaHXFwbFobYmLY7pTQ2Z5fKgUhtAFVBfwkHYnf0jbwE066X4dcq1NRG6jg8ht7BBynZ2nkMOE4JXYYaL52qVd4faTl3n2cosW5oRfZ0s7HaEgFrAN8bt64WWd9HKgArPJNTfkyVS1EQZVfDi6H3LOD5ztDhvZ5UNZbMq4WVWYAjP1Ex0oUQ/SpMbZ5sqA+KaoXrbdps03/GEkGikenN/Fg+CtZrN0Enzfg7bic556jadd57DXL67kIPQrhpPuXSofS1+BhE1pX5Gh6He3ymlCQrTCc1KdiEL/2GAv3Spk6kGw/2aXSKEumyBJ8rwv30vLxCjfRiZqmSmHLjRL6+WWDQEpyIXxTOaF8iCsSsUAQ6qvj8ruwGP3FXpB8X0ZH5jlKM7l70F59NwI9nbKqecuF+O6G6YxJ7Sg97iT2V1Lg3Ec+elavcwBpRfGEnoufugfnJz1xd7xz+L486cf+mel4o08B+HBF54Nz7Vz2j9r4HXNov/T3tFnaS03b5kq2YrqL2n8k1udF0hJBk60xfdWr6LFy+BQoi19e/szvZWuQWW8EPsnn48vzjnBczGioJFP+1mCZdyjQZlx05iTG68b21aVepgmKh5mRa9BuHiOhB5an8vkumwOVBqy0ncigyqQS2gBs0YmYIIV6MHDDdscc2e0H6EkrEP0ahLpV6U5VJOuDypobNH1ilSRRQPJmDUKCJdapXcvapZKvqDd8nBX5Dd+FfNyLhr6//MT9uTT6d7Z4TkeEM2dhqw4bV3u60MuIr/nPJYOtD5wBP7jc7zDWy9aUSt5oeSsBDmWkBKnVLzLNS4j6VwxNHyaVlyWKqoUMVTREVXfOVUMFRiUAMPmM9ot1QImqeayQndY2zRaZWXza/qjaq7skFHgNT1S1VZ2yecTRnJsGjqaqGJpY0UJISNEegI03j3Xx3w13cF3soelz7J771b1LwtzUkdUFfZ3cZIGdTr7/UBTASY8H4NF4ESwZ+ewEiZltkWiok7xdWybr1kxzKUCLxvmfLWVw5zvUrZTkhCyndLre2XWXdGvQpGX98ys+GIWziLcVvOwpNermbg4fkUuLg3fb8rGBP3lfFwZgKSO+PfKaSnK5JLNj9BMu7YiAmkNQ19UyTGWLMFZK8QnKThrDkYYGXklFBh5mP+lnTXjyZ5dOFfuwCSUnniQ+76Xb5zErwz+qNg50e0itYo4ZdpRA/h2F0dwEUZjCo4IYeeY6tQmFQBXBZRQrA9d/aeRkqwp1YnKfIXVwSe4j6foE//JyJMKeCoWpXTvLExoHZOSj0jRUcYV4LTvhnMv6khpGW5n3paSZhlj5M0alcTzMDEf6tbxHI/DUZTCtZfAzAoSHIaxBk6hcm7eq58Z/I6UXeMuCRfBMF4u5stFWpPhoBwJ8zVWjVeYNdiWssqqkbN6FG6JlFi6sxv4f23uYQxgusu3C9CeYxjf0GOWD5AsF6t98KYxJ7/TosSodXcRUzxMDk+JBmzCMS0qXqU5RVas1R+HyvVJYTa2uvPgwJumEqdcg5cP3377YKtTn3YPVHEjZSG6gx3x7be3jxVUkh7qW+wbjYzyS1fgnOFrF87QvgZLnR8J8Lx5GWKSJ16Gmir8+6LEfPkajIayygq8arkKyNnu4gsGAvEEWwRfFjV+Xy5/DiL783m5vP3h5Lj/V5WYIl3OZph5IZe6mmxf0O1VVrdcEvoKZ2FlTcPVVpXnHTObYmTFbY6ipslJ0fUxn80a9ajIbFLdY2kUk9MeSsm/TAxQfMjGZX6O23ohe+9tLnWvuK0/wwh8KVAFvokXDT3E43XmLluau+xcUm5ztOU4ktk6P97kO/WXs3lak4WY0aLFbltnCJl5IL6l6PJQW1Gh+O5ecrXEXGSn+JTU/ECb5neHQz8eD4fSzjJ3Pd8ferI4bKXN4we2QzdvhfDGkKEVdYxDBzYKejoUvkvRX6uqwDLzYvBR3CDSQAWPMAPSLuIESAW15JSA8hjmA7WJAlgfLwQ1TpPjOlbD1y48181ITnoZxUNqpGce0n7pukIDi808NYXPPh8fHx7/SLM4i7EsxZ9meBpvdcalzEOxS8ug6gdVMD+jKpKBMr7Uq4JWc3CJ+RAvE5kcBIoKpV5JxNjWCnXzQYK5sEAdGZG7u9qScUJ6nc1JNVY3LgsRRHT/Byb2rasMvyx20I6L6bZlEvNBLjSgct1/VDgVj0G8o1xk+qoCR18W8Mkt31HBNGAQu3ymxK3ITUGQ6lZJ7tA1FKUgMnltLsGBhT/L9Yy3FDly4ZZowYYHdoXqLmUZimfre2B4HcXEVLRwwR9msrLq+4HUrREsrDkokS4DqYqg05dNUJSScbcEwJVXShjwaKf/PEAs9izEzNzwWhT1vRUEka+r0CDVPv85oKrcC8CqUSjeNyHx6km+ydI+6W9VF7ZoQq6siB8ramYEq6iZfVxRU/e3uq76nK/9KJNCEe8Vul8EpLMfFXvviOe6tRJUqVeOeApZnVYIbd8ZGhwi+p0YNVV87X/rj7l42nGuKrWg6vrFunzww7y5StkHtSApJM5zAAFHYMI65StpS5hGNzWubd1e9jWPrFlbU0Tjm6+vvldhzDOwEuU24Az/jeE/vy1XgdEQrT14qgsF2GUxcwuL7nG5kJmuRd5exqYZ3biRd5GqOxKMzHRIJgQ8vaApjWlU1N8YZaX7k32Ah2LqItkMlJF/QVVMVaQatfE+JW65WDWzcw0p0bquhZkC6Q+9LBVPuWbnDWEqsVmHhUtrmIZ4DEJRQx6nI9cLw34vYavy+gDRU5W+z1Vqy0YyJuUa/HUj/xUZwviM8RVZo9kyp3oHgtbOnXzigCmMgAojvn7CS+9neIjzvnC+DBfRymg0Skl7cnp6cn540TfOEtbQ5Mfef4qGSgLOTkuJItGpEj8FkC7WQGuYMv7CDoYVSR6F6r7lz4Y90zkEVo1AuuLc5BPnykwsJZ/gCPNoViObnYujwI8wXU0tJKW05BJhqtHmk2QYZpG5rDDp79H5SeGkIBlODDfHqnHwIsPntaKLNCYb/yPdxIKGj0OdnTQPZb50nHXn0hXTJjs+2Xju/DBSpbpBvDRH5wcDuZI7W6y6bsYH9FYAqmbq3BnILL8tc28dDZkvYHjogB+gOZcP82dndZ0sjQTT+1lYpTOpto53pIPhu8Uwr7wJVBo/nSznIas30rD5VOIxZcyQe6aX7ETzlln49zdBhrumg8rU7ZQUQmyYp55GjctP7L9Gd0kMi8kiFg+A4GNPZKYvUt1ydlCnNEATUi8LhSpsk7qTq8CUTHSOyJtZLMuCLg2HmH9tOKSDJ8MhmlKGQ3mHLNtV3hJDvf3efm+/t9/b7+339nv7vf3efm+/t9/b7+339nv7/eP8/j+dHqISAMgAAA=="
tarfile.open(fileobj=io.BytesIO(base64.b64decode(_PKG)), mode="r:gz").extractall("/content/code")
print("unpacked to /content/code:")
for f in sorted(os.listdir("/content/code")):
    print("   ", f)
print("\nno pip installs needed: the exact tests (McNemar, Fisher) are")
print("implemented in stats.py, and pandas/sklearn/numpy are preinstalled.")

## Cell 2 — mount Drive and verify the frozen reference is there

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# ---- Drive layout. Edit DRIVE_ROOT only; everything else follows. ----------
DRIVE_ROOT = "/content/drive/MyDrive/CardioSentry"
RUN_NAME   = "af_lvh_joint_v1"

HYBRID_RUNS = f"{DRIVE_ROOT}/cardiosentry_runs/hybrid_v1"
REFERENCE   = f"{HYBRID_RUNS}/resnet50_hires"         # the four-head model, READ-ONLY
RUNS        = f"{DRIVE_ROOT}/cardiosentry_runs/{RUN_NAME}"   # THIS round writes here

os.makedirs(RUNS, exist_ok=True)

print("Drive layout check:")
for _p, _label in (
        (REFERENCE + "/predictions/test_predictions.csv", "frozen test predictions"),
        (REFERENCE + "/result.json",                     "reference result.json (optional)")):
    print(f"  {'FOUND ' if os.path.exists(_p) else 'missing'}  {_label:<32} {_p}")

if not os.path.exists(REFERENCE + "/predictions/test_predictions.csv"):
    raise SystemExit(
        f"\nMISSING the frozen predictions file:\n  "
        f"{REFERENCE}/predictions/test_predictions.csv\n"
        f"This round reads the EXACT file the reference numbers and the\n"
        f"previous subgroup analysis were computed from. Copy the\n"
        f"hybrid_v1/resnet50_hires/ run folder back to Drive if it is missing.")

## Cell 3 — THE ANALYSIS ⚠️ the deliverable (~10 s, CPU)

Refuses to print anything unless the frozen predictions match the reference
(cross-checks listed in the notebook header). Then prints the per-head
tables, the joint breakdown, the comparisons with exact tests and CIs, and
the verdict. Everything is also written to
`cardiosentry_runs/af_lvh_joint_v1/` on Drive.

In [ ]:
%cd /content/code
!python analyze.py     --predictions "$REFERENCE/predictions/test_predictions.csv"     --result-json  "$REFERENCE/result.json"     --out          "$RUNS"

## Cell 4 — the verdict, isolated

In [ ]:
%cd /content/code
_report = open(f"{RUNS}/report.txt").read()
_i = _report.find("VERDICT:")
print(_report[_i:] if _i >= 0 else _report[-4000:])

## Cell 5 — morning check

In [ ]:
!cat "$RUNS/STATUS.txt"
!ls "$RUNS"

## Where the results live

```
/content/drive/MyDrive/CardioSentry/cardiosentry_runs/af_lvh_joint_v1/
├── STATUS.txt               DONE
├── report.txt               the whole analysis + verdict      <- read this
├── summary.json             model, thresholds, tests, verdict
├── af_head_rows.csv         AF head per subgroup (TP/FP/FN/TN + metrics)
├── lvh_head_rows.csv        LVH head per subgroup
├── joint_breakdown.csv      4 joint categories + bootstrap CIs
├── recalls.csv              recall point estimates + CIs
└── recall_deltas.csv        AF+LVH vs AF-only / LVH-only deltas + CIs
```

All on Drive, so it survives the runtime being recycled. The frozen
`hybrid_v1/` and `af_lvh_subgroup_v1/` folders are untouched — this round
only reads them.

**Read in this order:** report.txt → cross-checks → the two per-head tables →
the joint breakdown → the comparisons → the verdict → the limitation
paragraph.